# Datenanalyse mit SQL & Python - Tag 2: Übungen

**Teil A:** 30-Minuten Refresh & Warm-up mit Fitness-Bookings  
**Teil B:** Fitness-Daten als Transferübung zu Tag 2  
**Ziel:** Erst Tag 1 mit `bookings` wiederholen, dann `HAVING`, `JOIN`, Subqueries und CTEs mit dem Fitness-Datenmodell anwenden.

## Zeitplanung der Übungen

| Teil | Fokus | Geschätzte Zeit |
|---|---|---|
| Teil A | Tag-1-Refresh mit `bookings` | ca. 30 Minuten |
| Teil B | Transferübung zu Tag 2 mit `bookings`, `members`, `classes` | ca. 45-60 Minuten |

Die Zeiten sind Richtwerte für Gruppenarbeit. Schnellere Gruppen können die Reflexionsfragen vertiefen.


## Einrichtung & Daten laden

Führe diese Zelle zuerst aus. Sie lädt die Fitness-Daten in eine SQLite-Datenbank.


In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


def find_fitness_dir():
    candidates = [
        Path('data_practice_tables/fitness'),
        Path('Tag 1/data_practice_tables/fitness'),
        Path('../Tag 1/data_practice_tables/fitness'),
    ]
    for candidate in candidates:
        if (candidate / 'fitness_bookings.csv').exists():
            return candidate
    raise FileNotFoundError('Fitness-Daten wurden nicht gefunden.')

FITNESS_DIR = find_fitness_dir()

fitness_bookings_df = pd.read_csv(FITNESS_DIR / 'fitness_bookings.csv')
fitness_members_df = pd.read_csv(FITNESS_DIR / 'fitness_members.csv')
fitness_classes_df = pd.read_csv(FITNESS_DIR / 'fitness_classes.csv')

conn = sqlite3.connect(':memory:')
fitness_bookings_df.to_sql('bookings', conn, index=False, if_exists='replace')
fitness_members_df.to_sql('members', conn, index=False, if_exists='replace')
fitness_classes_df.to_sql('classes', conn, index=False, if_exists='replace')

print('Geladene Tabellen: bookings, members, classes')


## Teil A: 30-Minuten Refresh & Warm-up mit Fitness-Bookings (ca. 30 Minuten)

In Teil A arbeitest du **nur mit der Tabelle `bookings`**.

Das ist die Wiederholung von Tag 1:

- `SELECT`
- `WHERE`
- `COUNT`
- `GROUP BY`
- `ORDER BY`
- `LIMIT`

Die Tabelle `bookings` enthält technische IDs:

| Spalte | Bedeutung |
|---|---|
| `booking_id` | eindeutige Buchungs-ID |
| `member_id` | ID des Mitglieds |
| `class_id` | ID des gebuchten Kurses |

**Wichtig:** In Teil A geht es bewusst noch nicht um Namen, Level oder Trainer. Diese Informationen kommen erst später mit `JOIN` dazu.


### Übung A1: Erste Buchungen anzeigen (ca. 3 Minuten)

**Aufgabe:** Zeige alle Spalten aus `bookings` an.


In [ ]:
# Aufgabe: Zeige alle Spalten aus bookings an.
query = '''
SELECT _____
FROM _____;
'''

result = pd.read_sql_query(query, conn)
result.head(10)


### Übung A2: Nur ausgewählte Spalten anzeigen (ca. 3 Minuten)

**Aufgabe:** Zeige nur `member_id` und `class_id` an.

**Reflexion:** Welche fachlichen Informationen fehlen, wenn wir nur IDs sehen?


In [ ]:
# Aufgabe: Wähle member_id und class_id aus.
query = '''
SELECT _____, _____
FROM bookings;
'''

result = pd.read_sql_query(query, conn)
result.head(10)


### Übung A3: Anzahl aller Buchungen (ca. 3 Minuten)

**Aufgabe:** Wie viele Buchungen gibt es insgesamt?

**Hinweis:** Verwende `COUNT(*)`.


In [ ]:
# Aufgabe: Zähle alle Buchungen.
query = '''
SELECT COUNT(*) AS anzahl_buchungen
FROM _____;
'''

result = pd.read_sql_query(query, conn)
result


### Übung A4: Buchungen eines Mitglieds filtern (ca. 4 Minuten)

**Aufgabe:** Zeige alle Buchungen von Mitglied `15`.

**Hinweis:** Verwende `WHERE member_id = 15`.


In [ ]:
# Aufgabe: Filtere Buchungen von member_id 15.
query = '''
SELECT *
FROM bookings
WHERE _____ = _____;
'''

result = pd.read_sql_query(query, conn)
result


### Übung A5: Buchungen eines Kurses zählen (ca. 4 Minuten)

**Aufgabe:** Wie oft wurde Kurs `202` gebucht?

**Hinweis:** Kombiniere `COUNT(*)` mit `WHERE class_id = 202`.


In [ ]:
# Aufgabe: Zähle Buchungen für class_id 202.
query = '''
SELECT COUNT(*) AS anzahl_buchungen
FROM bookings
WHERE _____ = _____;
'''

result = pd.read_sql_query(query, conn)
result


### Übung A6: Buchungen pro Kurs (ca. 5 Minuten)

**Aufgabe:** Wie viele Buchungen gibt es pro `class_id`?

**Hinweis:** Verwende `GROUP BY class_id`.


In [ ]:
# Aufgabe: Berechne Buchungen pro Kurs.
query = '''
SELECT
    class_id,
    COUNT(*) AS anzahl_buchungen
FROM bookings
GROUP BY _____
ORDER BY anzahl_buchungen DESC;
'''

result = pd.read_sql_query(query, conn)
result.head(10)


### Übung A7: Aktivste Mitglieder (ca. 5 Minuten)

**Aufgabe:** Welche fünf Mitglieder haben am häufigsten Kurse gebucht?

**Hinweis:** Gruppiere nach `member_id`, sortiere absteigend und nutze `LIMIT 5`.


In [ ]:
# Aufgabe: Finde die aktivsten Mitglieder nach ID.
query = '''
SELECT
    member_id,
    COUNT(*) AS anzahl_buchungen
FROM bookings
GROUP BY _____
ORDER BY _____ DESC
LIMIT _____;
'''

result = pd.read_sql_query(query, conn)
result


### Übung A8: Vorbereitung auf Tag 2 mit HAVING (ca. 3 Minuten)

**Aufgabe:** Welche Kurse wurden mindestens 3-mal gebucht?

**Hinweis:** Diese Aufgabe nutzt bereits `HAVING`, weil die Bedingung auf einer gruppierten Kennzahl liegt.


In [ ]:
# Aufgabe: Nutze HAVING für Kurse mit mindestens 3 Buchungen.
query = '''
SELECT
    class_id,
    COUNT(*) AS anzahl_buchungen
FROM bookings
GROUP BY class_id
HAVING _____ >= 3
ORDER BY anzahl_buchungen DESC;
'''

result = pd.read_sql_query(query, conn)
result


## Kurze Reflexion nach Teil A

Diskutiert in der Gruppe:

1. Welche Fragen konnten wir nur mit `bookings` beantworten?
1. Welche Fragen bleiben offen, solange wir nur IDs sehen?
1. Warum brauchen wir für Kursnamen, Trainer oder Mitgliedslevel einen `JOIN`?


## Teil B: Fitness-Daten als Transferübung (ca. 45-60 Minuten)

In Teil B arbeitest du weiter mit dem Fitness-Datenmodell:

- `bookings`
- `members`
- `classes`

Die SQL-Logik ist ähnlich wie im Tag-2-Hauptnotebook mit Shop-Daten. Ziel ist, die Konzepte auf ein anderes Datenmodell zu übertragen:

- `HAVING`
- `JOIN`
- einfache Subqueries
- CTEs mit `WITH`
- Primärschlüssel und Fremdschlüssel im Datenmodell erkennen


### Übung B1: Datenmodell verstehen (ca. 5 Minuten)

**Aufgabe:** Schau dir die drei Tabellen kurz an.

**Reflexion:** Welche Spalten sind IDs? Welche IDs verbinden die Tabellen miteinander?


In [ ]:
# Aufgabe: Schaue dir die drei Tabellen kurz an.
print('bookings')
display(pd.read_sql_query('SELECT * FROM bookings LIMIT 5;', conn))

print('members')
display(pd.read_sql_query('SELECT * FROM members LIMIT 5;', conn))

print('classes')
display(pd.read_sql_query('SELECT * FROM classes LIMIT 5;', conn))


### Übung B2: Buchungen pro Mitglied mit HAVING (ca. 5 Minuten)

**Aufgabe:** Welche Mitglieder haben mindestens 3 Buchungen?

**Hinweis:** Die Bedingung bezieht sich auf `COUNT(*)`, also brauchst du `HAVING`.


In [ ]:
# Aufgabe: Filtere Gruppen mit HAVING.
query = '''
SELECT
    member_id,
    COUNT(*) AS anzahl_buchungen
FROM bookings
GROUP BY member_id
HAVING _____ >= 3
ORDER BY anzahl_buchungen DESC;
'''

result = pd.read_sql_query(query, conn)
result


### Übung B3: JOIN zwischen Bookings und Members (ca. 6 Minuten)

**Aufgabe:** Zeige `booking_id`, Mitgliedsname und Level.

**Hinweis:** Verbinde `bookings` und `members` über `member_id`.


In [ ]:
# Aufgabe: Verbinde bookings mit members.
query = '''
SELECT
    b.booking_id,
    m.name,
    m.level
FROM bookings AS b
INNER JOIN members AS m
    ON b._____ = m._____
ORDER BY b.booking_id;
'''

result = pd.read_sql_query(query, conn)
result.head(10)


### Übung B4: JOIN mit Tag-1-Aggregation (ca. 6 Minuten)

**Aufgabe:** Wie viele Buchungen gibt es pro Mitgliedslevel?

**Hinweis:** Kombiniere `JOIN`, `GROUP BY`, `COUNT` und `ORDER BY`.


In [ ]:
# Aufgabe: Aggregiere Buchungen pro Level.
query = '''
SELECT
    m.level,
    COUNT(b.booking_id) AS anzahl_buchungen
FROM bookings AS b
INNER JOIN members AS m
    ON b.member_id = m.member_id
GROUP BY _____
ORDER BY _____ DESC;
'''

result = pd.read_sql_query(query, conn)
result


### Übung B5: Bookings und Classes joinen (ca. 6 Minuten)

**Aufgabe:** Zeige `booking_id`, Kursname, Trainerin oder Trainer und Preis.

**Hinweis:** Verbinde `bookings` und `classes` über `class_id`.


In [ ]:
# Aufgabe: Verbinde bookings mit classes.
query = '''
SELECT
    b.booking_id,
    c.class_name,
    c.trainer,
    c.price
FROM bookings AS b
INNER JOIN classes AS c
    ON b._____ = c._____
ORDER BY b.booking_id;
'''

result = pd.read_sql_query(query, conn)
result.head(10)


### Übung B6: Umsatz pro Trainerin oder Trainer (ca. 7 Minuten)

**Aufgabe:** Berechne pro Trainerin oder Trainer die Anzahl der Buchungen und den geschätzten Umsatz.

**Hinweis:** Jede Buchung zählt den Kurspreis einmal. Nutze `SUM(c.price)`.


In [ ]:
# Aufgabe: Berechne Buchungen und Umsatz pro Trainer.
query = '''
SELECT
    c.trainer,
    COUNT(b.booking_id) AS anzahl_buchungen,
    SUM(c.price) AS revenue
FROM bookings AS b
INNER JOIN classes AS c
    ON b.class_id = c.class_id
GROUP BY _____
ORDER BY revenue DESC;
'''

result = pd.read_sql_query(query, conn)
result


### Übung B7: Kurse mit mindestens 3 Buchungen (ca. 7 Minuten)

**Aufgabe:** Zeige Kursname, Trainerin oder Trainer und Buchungsanzahl für Kurse mit mindestens 3 Buchungen.

**Hinweis:** Hier brauchst du `JOIN`, `GROUP BY` und `HAVING`.


In [ ]:
# Aufgabe: Nutze HAVING nach der Gruppierung.
query = '''
SELECT
    c.class_name,
    c.trainer,
    COUNT(b.booking_id) AS anzahl_buchungen
FROM bookings AS b
INNER JOIN classes AS c
    ON b.class_id = c.class_id
GROUP BY c.class_id, c.class_name, c.trainer
HAVING _____ >= 3
ORDER BY anzahl_buchungen DESC;
'''

result = pd.read_sql_query(query, conn)
result


### Übung B8: Vollständige Buchungsübersicht (ca. 6 Minuten)

**Aufgabe:** Erstelle eine Übersicht mit `booking_id`, Mitgliedsname, Level, Kursname, Trainerin oder Trainer und Preis.

**Hinweis:** Dafür brauchst du zwei `INNER JOIN`s.


In [ ]:
# Aufgabe: Verbinde alle drei Fitness-Tabellen.
query = '''
SELECT
    b.booking_id,
    m.name,
    m.level,
    c.class_name,
    c.trainer,
    c.price
FROM bookings AS b
INNER JOIN members AS m
    ON b.member_id = m.member_id
INNER JOIN classes AS c
    ON b.class_id = c.class_id
ORDER BY b.booking_id;
'''

result = pd.read_sql_query(query, conn)
result.head(10)


### Übung B9: Subquery (ca. 8 Minuten)

**Aufgabe:** Finde Mitglieder, die mehr Buchungen haben als der Durchschnitt pro aktivem Mitglied.

**Hinweis:** Die innere Abfrage berechnet zuerst die Buchungsanzahl pro Mitglied.


In [ ]:
# Aufgabe: Nutze eine Unterabfrage für den Durchschnitt.
query = '''
SELECT
    member_id,
    COUNT(*) AS anzahl_buchungen
FROM bookings
GROUP BY member_id
HAVING COUNT(*) > (
    SELECT AVG(buchungsanzahl)
    FROM (
        SELECT COUNT(*) AS buchungsanzahl
        FROM bookings
        GROUP BY member_id
    )
)
ORDER BY anzahl_buchungen DESC;
'''

result = pd.read_sql_query(query, conn)
result


### Übung B10: CTE als Vergleich (ca. 8 Minuten)

**Aufgabe:** Löse B9 noch einmal mit einer CTE.

**Vergleich:** Welche Version ist für dich leichter zu lesen: Subquery oder CTE?


In [ ]:
# Aufgabe: Nutze WITH, um Zwischenergebnisse lesbarer zu machen.
query = '''
WITH member_totals AS (
    SELECT
        member_id,
        COUNT(*) AS anzahl_buchungen
    FROM bookings
    GROUP BY member_id
),
average_total AS (
    SELECT AVG(anzahl_buchungen) AS durchschnitt
    FROM member_totals
)
SELECT
    mt.member_id,
    mt.anzahl_buchungen
FROM member_totals AS mt
CROSS JOIN average_total AS avg_t
WHERE mt.anzahl_buchungen > avg_t.durchschnitt
ORDER BY mt.anzahl_buchungen DESC;
'''

result = pd.read_sql_query(query, conn)
result


## Abschlussreflexion

Diskutiert in der Gruppe:

1. Was war in Teil A reine Tag-1-Wiederholung?
1. Welche Aufgabe in Teil B war ähnlich wie im Shop-Beispiel aus dem Hauptnotebook?
1. Was ist der Unterschied zwischen `WHERE` und `HAVING`?
1. Wann ist eine CTE lesbarer als eine Subquery?
